In [ ]:
# ----------------------------- Setup ----------------------------- #
%matplotlib tk
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.widgets import Button
import re

prec = 500
RHP = RealField(prec)
CHP = ComplexField(prec)

# ----------------------------- Printing precision -----------------------------
print_digits = 20          # ← number of decimal digits shown for roots

# ----------------------------- Input ----------------------------- #
def get_polynomial_coeffs():
    """
    Read polynomial coefficients from user:
    - Accepts integers, decimals, scientific notation
    - Strips leading zeros
    - Rejects all-zero polynomials
    - RETURNS BOTH: high-precision values + original string representations
      (so the printed polynomial uses EXACTLY the digits the user typed)
    """
    pattern = re.compile(r'^[+-]?(\d+(\.\d*)?|\.\d+)([eE][+-]?\d+)?$')
   
    while True:
        s = input("Enter polynomial coefficients (highest degree first): ").strip()
        if not s:
            print("Error: No input provided.")
            continue
        tokens = s.split()
        coeffs_hp = []
        coeff_strings = []          # keep exact user input strings
        for t in tokens:
            t_clean = t.lstrip("0") or "0"
            if t_clean.startswith("."):
                t_clean = "0" + t_clean
            if not pattern.fullmatch(t_clean):
                print(f"Invalid number: '{t}'")
                break
            try:
                coeffs_hp.append(RHP(t_clean))
                coeff_strings.append(t)   # store EXACTLY what the user typed
            except Exception:
                print(f"Could not convert '{t}' to high-precision number.")
                break
        else:
            # Reject all-zero polynomial
            if all(c == 0 for c in coeffs_hp):
                print("Polynomial cannot be identically zero.")
                continue
            # Strip leading zeros to get true degree
            while coeffs_hp and coeffs_hp[0] == 0:
                coeffs_hp.pop(0)
                coeff_strings.pop(0)
            return coeffs_hp, coeff_strings   # ← returns TWO lists

coeffs_hp, coeff_strings = get_polynomial_coeffs()

# ----------------------------- Polynomial Construction ----------------------------- #
def build_polynomial_from_coeffs(coeffs_hp, RHP, CHP):
    """
    Safely construct polynomial from coefficients (highest degree first),
    avoiding exponent/order bugs.
    """
    R.<x> = RHP[]
    
    # Use Horner's method (robust, order-safe)
    f = R(0)
    for c in coeffs_hp:
        f = f * x + c
    
    fC = f.change_ring(CHP)
    return f, fC


# build polynomial
f, fC = build_polynomial_from_coeffs(coeffs_hp, RHP, CHP)

# ----------------------------- Clean Printing (using user strings) ----------------------------- #
def polynomial_to_string(coeffs_hp, coeff_strings, var="x"):
    if not coeff_strings:
        return "0"
    
    terms = []
    n = len(coeff_strings)

    for i, (c_str, c_hp) in enumerate(zip(coeff_strings, coeffs_hp)):
        power = n - i - 1

        # skip numerical zeros
        if abs(c_hp) < RHP('1e-30'):
            continue

        sign = "-" if c_str.startswith("-") else "+"
        fc_abs = c_str.lstrip("+-")

        # --- FIX: handle coefficient properly ---
        if power == 0:
            term = fc_abs
        elif power == 1:
            if fc_abs == "1":
                term = f"{var}"
            else:
                term = f"{fc_abs}*{var}"
        else:
            if fc_abs == "1":
                term = f"{var}^{power}"   # ← FIX HERE
            else:
                term = f"{fc_abs}*{var}^{power}"

        terms.append((sign, term))

    if not terms:
        return "0"

    # first term
    first_sign, first_term = terms[0]
    result = first_term if first_sign == "+" else f"-{first_term}"

    for sign, term in terms[1:]:
        result += f" {sign} {term}"

    return result

poly_str = polynomial_to_string(coeffs_hp, coeff_strings)
print(f"\nPolynomial: f(x) = {poly_str} = 0")

# ----------------------------- Roots ----------------------------- #
roots = fC.roots(multiplicities=True) if len(coeffs_hp) > 1 else []

# ----------------------------- Helpers ----------------------------- #
def poly_eval(coeffs, r):
    p = CHP(0)
    for c in coeffs:
        p = p * r + c
    return p

def relative_residual(coeffs, r):
    numerator = abs(poly_eval(coeffs, r))
    denom = sum(abs(c) * abs(r)^(len(coeffs) - i - 1) for i, c in enumerate(coeffs))
    return numerator / denom if denom != 0 else numerator

# ----------------------------- Print Roots ----------------------------- #
print(f"\nPolynomial degree: {len(coeffs_hp)-1}")
if roots:
    print("Roots with high precision, multiplicity and relative residuals:")
    for i, (r, mult) in enumerate(roots, 1):
        root_str = r.n(digits=print_digits)
        res = relative_residual(coeffs_hp, r)
        print(f"Root {i}: {root_str} (multiplicity {mult})   Residual: {res:.2e}")
else:
    print("No roots (constant polynomial)")

# ----------------------------- Plotting ----------------------------- #
if roots:
    roots_np = np.array([complex(r) for r, _ in roots])
    multiplicities = [mult for _, mult in roots]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

    # ---- Complex Plane ----
    ax1.axhline(0, color='gray', lw=1)
    ax1.axvline(0, color='gray', lw=1)
    for r, mult in zip(roots_np, multiplicities):
        ax1.scatter(r.real, r.imag, color='red', s=10*mult)
       
    ax1.set_title("Roots in Complex Plane (size = multiplicity)")
    ax1.set_xlabel("Re")
    ax1.set_ylabel("Im")
    ax1.grid(True)
    ax1.set_aspect('auto')

    # ---- Real Line Plot (with fixes for high-degree / high-scaling) ----
    real_roots = [(r, mult) for (r, mult) in roots if abs(r.imag()) < RHP('1e-12')]

    if roots_np.size > 0:
        xmin = float(min(roots_np.real)) - 1.0
        xmax = float(max(roots_np.real)) + 1.0
    else:
        xmin, xmax = -5.0, 5.0

    x_vals_initial = np.linspace(xmin, xmax, 2000)
    x_list = x_vals_initial.tolist()

    for r, _ in real_roots:
        root_float = float(r.real())
        if not any(abs(root_float - x) < 1e-12 for x in x_list):
            x_list.append(root_float)
    x_list.sort()
    x_vals = np.array(x_list)

    y_vals = np.array([float(f(RHP(xx))) for xx in x_vals])

    max_abs = np.max(np.abs(y_vals)) if len(y_vals) > 0 else 1.0
    if np.isinf(max_abs) or max_abs > 1e300:
        print("NOTICE: Polynomial values exceed float range → clipping plot")
        y_vals = np.clip(y_vals, -1e300, 1e300)
        max_abs = 1e300

    ax2.plot(x_vals, y_vals, lw=1, label="f(x)")
    ax2.axhline(0, color='gray', lw=1)

    for r, mult in real_roots:
        ax2.scatter(float(r.real()), 0.0, color='red', s=10*mult, zorder=5)

    ax2.set_title("Polynomial on Real Line (size = multiplicity)")
    ax2.set_xlabel("x")
    ax2.set_ylabel("f(x)")
    ax2.grid(True)
    ax2.legend()

    init_ylim = ax2.get_ylim()

    # ----------------------------- Buttons -----------------------------
    linthresh = 1.0

    def set_linear(event):
        ax2.set_yscale('linear')
        ax2.set_ylim(init_ylim)
        fig.canvas.draw_idle()

    def set_symlog(event):
        ax2.set_yscale('symlog', linthresh=linthresh, linscale=1.0)
        print("→ Using symmetric symlog scale")
        fig.canvas.draw_idle()

    axlinear = plt.axes([0.8, 0.02, 0.1, 0.04])
    axsymlog = plt.axes([0.65, 0.02, 0.1, 0.04])
    b_linear = Button(axlinear, 'Linear')
    b_symlog = Button(axsymlog, 'Symlog')
    b_linear.on_clicked(set_linear)
    b_symlog.on_clicked(set_symlog)

    plt.tight_layout()
    plt.show()

else:
    print("No plot: polynomial is constant, no roots to display.")